<h1>C4 Solution (SageMaker Python SDK v3)</h1>

This notebook is the SageMaker Python SDK **v3** version of the original course solution.
Main API changes compared to the v2 notebook:

| SDK v2 | SDK v3 |
|---|---|
| `sagemaker.pytorch.PyTorch` (Estimator) + `.fit()` | `sagemaker.train.ModelTrainer` + `.train()` |
| `framework_version` / `py_version` on the estimator | explicit `training_image` from `sagemaker.core.image_uris.retrieve()` |
| `sagemaker.tuner.HyperparameterTuner(estimator, ...)` + `.fit()` | `sagemaker.train.tuner.HyperparameterTuner(model_trainer=...)` + `.tune()` |
| `tuner.best_estimator()` | `TrainingJob.get(tuner.best_training_job())` |
| `sagemaker.analytics.HyperparameterTuningJobAnalytics` | `tuner.analytics()` (`sagemaker.core.analytics`) |
| `Estimator.attach(job_name)` | `sagemaker.core.resources.TrainingJob.get(job_name)` |
| `estimator.model_data` | `training_job.model_artifacts.s3_model_artifacts` |
| `PyTorchModel(...)` + `.deploy()` returning a `Predictor` | `sagemaker.serve.ModelBuilder` `.build()` / `.deploy()` returning an `Endpoint` |
| `predictor.predict(data, initial_args={...})` | `endpoint.invoke(body=..., content_type=...)` |
| Debugger `rules` / `debugger_hook_config` / `profiler_config` | not supported by `ModelTrainer` (see the note further down) |

The PyTorch container was also upgraded from 1.4 (no longer available) to 2.6 / Python 3.12.

<h3>Get the data</h3>

In [ ]:
%%capture
!wget https://s3-us-west-1.amazonaws.com/udacity-aind/dog-project/dogImages.zip

In [ ]:
%%capture
!unzip dogImages.zip

<h3>Install and import</h3>

In [ ]:
%%capture
!pip install -U "sagemaker>=3,<4" pillow requests numpy

In [ ]:
import json
import os
import shutil
import time

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.parameter import CategoricalParameter, ContinuousParameter
from sagemaker.core.resources import TrainingJob
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, MetricDefinition, SourceCode, StoppingCondition
from sagemaker.train.tuner import HyperparameterTuner

<h3>Session, S3 location and container images</h3>

The original solution used a hard-coded bucket (`s3://udacitysolution/`) that only exists in the
instructor's account, so this version uses your SageMaker default bucket instead.

SDK v3 no longer resolves the container from `framework_version`/`py_version`; you pass an image URI explicitly.

In [ ]:
session = Session()
role = get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
data_s3_uri = f"s3://{bucket}/dogImages/"

framework_version = "2.6.0"
py_version = "py312"

training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
    image_scope="training",
)
inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=framework_version,
    py_version=py_version,
    instance_type="ml.m5.large",
    image_scope="inference",
)
print(data_s3_uri)
print(training_image)
print(inference_image)

<h3>Copy the data to S3</h3>

In [ ]:
%%capture
!aws s3 cp dogImages {data_s3_uri} --recursive

<h3>Stage the training and inference scripts</h3>

`SourceCode(source_dir=...)` uploads the **whole** directory, and ModelBuilder packs the whole inference
`source_dir` into `model.tar.gz`. Copying each script into its own small folder keeps the
`dogImages` dataset from being uploaded with the code.

In [ ]:
os.makedirs("train_src", exist_ok=True)
shutil.copy("hpo.py", "train_src/hpo.py")

os.makedirs("inference_src", exist_ok=True)
shutil.copy("infernce2.py", "inference_src/infernce2.py")
# infernce2.py downloads images from URLs; the inference container installs this at startup
with open("inference_src/requirements.txt", "w") as f:
    f.write("requests\n")

<h3>Set up parameters, ModelTrainer, and tuner</h3>

In [ ]:
hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.001, 0.1),
    "batch_size": CategoricalParameter([32, 64, 128, 256, 512]),
}

objective_metric_name = "Test Loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "Test Loss", "Regex": "Testing Loss: ([0-9\\.]+)"}]

In [ ]:
hpo_trainer = ModelTrainer(
    training_image=training_image,
    source_code=SourceCode(source_dir="train_src", entry_script="hpo.py"),
    compute=Compute(instance_type="ml.m5.xlarge", instance_count=1),
    # Fixed (not tuned) hyperparameter: one epoch per tuning job keeps HPO cheap
    hyperparameters={"epochs": 1},
    base_job_name="pytorch-dog-hpo",  # job names may not contain underscores
    role=role,
    sagemaker_session=session,
)

tuner = HyperparameterTuner(
    model_trainer=hpo_trainer,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=metric_definitions,
    max_jobs=2,
    max_parallel_jobs=2,
    objective_type=objective_type,
    base_tuning_job_name="dog-hpo",
)

<h3>Run the tuner</h3>

The v2 notebook set `SM_CHANNEL_TRAINING` / `SM_MODEL_DIR` / `SM_OUTPUT_DATA_DIR` locally before fitting.
That had no effect on the remote job (SageMaker sets them inside the container), so it was removed.

In [ ]:
training_input = InputData(channel_name="training", data_source=data_s3_uri)
tuner.tune(inputs=[training_input], wait=True)

<h3>Describe the tuning results</h3>

In [ ]:
jobs = tuner.analytics().dataframe()

jobs.sort_values('FinalObjectiveValue', ascending=0)

## Imp: If kernel dies, how to continue from a completed training job

In [ ]:
#BetterTrainingJobName='pytorch-dog-hpo-...'

In [ ]:
#best_training_job = TrainingJob.get(training_job_name=BetterTrainingJobName)

In [ ]:
#best_training_job.hyper_parameters

In [ ]:
#model_location = best_training_job.model_artifacts.s3_model_artifacts

<h3>Prepare to perform Training on the best training job</h3>

In [ ]:
best_training_job = TrainingJob.get(
    training_job_name=tuner.best_training_job(),
    session=session.boto_session,
)

In [ ]:
best_training_job.hyper_parameters

In [ ]:
hyperparameters = {"batch_size": int(best_training_job.hyper_parameters['batch_size'].strip('"')), \
                   "learning_rate": best_training_job.hyper_parameters['learning_rate'].strip('"'), \
                   "epochs": 1}  # 5}   # hpo.py also stops early once validation loss stops improving
hyperparameters

<h3>About SageMaker Debugger and Profiler</h3>

The v2 solution attached Debugger rules (`vanishing_gradient`, `overfit`, `overtraining`,
`poor_weight_initialization`, `ProfilerReport`) plus `DebuggerHookConfig` / `ProfilerConfig` to the estimator.

These were dropped in this version because:

* `ModelTrainer` in SDK v3 has no `rules`, `debugger_hook_config` or `profiler_config` parameters.
* The rules depend on the `smdebug` hook, which does not support PyTorch 2.x. AWS deprecated Debugger
  framework profiling starting with PyTorch 2.0, so the rules would get no tensors to analyse on the
  2.6 container anyway.

As a lightweight replacement, the trainer below publishes `Test Loss` to CloudWatch through
`with_metric_definitions`. For deeper insight, log to TensorBoard from the script and use
`ModelTrainer.with_tensorboard_output_config(...)`.

<h2>Creating a ModelTrainer</h2>

In [ ]:
#adjust this cell to accomplish multi-instance training
trainer = ModelTrainer(
    training_image=training_image,
    source_code=SourceCode(source_dir="train_src", entry_script="hpo.py"),
    compute=Compute(instance_type="ml.m5.xlarge", instance_count=1),
    hyperparameters=hyperparameters,
    # ModelTrainer stops jobs after 1 hour by default; one CPU epoch takes ~20 minutes
    stopping_condition=StoppingCondition(max_runtime_in_seconds=4 * 60 * 60),
    base_job_name="dog-pytorch",
    role=role,
    sagemaker_session=session,
).with_metric_definitions([
    MetricDefinition(name="Test Loss", regex="Testing Loss: ([0-9\\.]+)"),
])

In [ ]:
trainer.train(input_data_config=[training_input], wait=False)

<h2>Creating a ModelTrainer - Multi-Instance Training</h2>

In [ ]:
###in this cell, create and train a ModelTrainer using multi-instance training
# Hint (SDK v3): the instance count now lives on the Compute config, e.g.
# compute=Compute(instance_type="ml.m5.xlarge", instance_count=2)

<h2>Deployment</h2>

In [ ]:
training_job = trainer._latest_training_job
training_job.wait()
training_job.refresh()
model_location = training_job.model_artifacts.s3_model_artifacts
model_location

`ModelBuilder` replaces `PyTorchModel` and the custom `ImagePredictor` class. With a SageMaker PyTorch
image and a `SourceCode` config, it repacks `infernce2.py` into `model.tar.gz` (under `code/`) and sets
`SAGEMAKER_PROGRAM`, the same as v2 did. `deploy()` returns a `sagemaker.core.resources.Endpoint`.

In [ ]:
from sagemaker.serve import ModelBuilder

endpoint_name = f"dog-breed-classifier-{int(time.time())}"

model_builder = ModelBuilder(
    s3_model_data_url=model_location,
    image_uri=inference_image,
    source_code=SourceCode(source_dir="inference_src", entry_script="infernce2.py"),
    role_arn=role,
    sagemaker_session=session,
    instance_type="ml.m5.large",
)
model = model_builder.build()

In [ ]:
endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type="ml.m5.large",
)
print(endpoint_name)  # set this as ENDPOINT_NAME on the Lambda function

In [ ]:
def predict(payload, content_type):
    """Replaces v2's ImagePredictor: sends payload and decodes the JSON response."""
    response = endpoint.invoke(body=payload, content_type=content_type, accept="application/json")
    body = response.body.read() if hasattr(response.body, "read") else response.body
    return json.loads(body)

In [ ]:
import requests
#request_dict={ "url": "https://cdn1-www.cattime.com/assets/uploads/2011/12/file_2744_british-shorthair-460x290-460x290.jpg" }
request_dict={ "url": "https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2017/11/20113314/Carolina-Dog-standing-outdoors.jpg" }

img_bytes = requests.get(request_dict['url']).content
type(img_bytes)

In [ ]:
from PIL import Image
import io
Image.open(io.BytesIO(img_bytes))

In [ ]:
response=predict(img_bytes, "image/jpeg")

In [ ]:
response2=predict(json.dumps(request_dict), "application/json")

In [ ]:
type(response2[0][0])

In [ ]:
response2[0]

In [ ]:
import numpy as np
np.argmax(response, 1)

<h2>Clean up</h2>

Endpoints are billed while running. Delete it when you are done (after the Lambda exercise, if you use it).

In [ ]:
#endpoint.delete()